In [2]:
from ingest import load_faq_data
documents = load_faq_data()

In [3]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [4]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

115

In [5]:
documents = documents_llm

In [6]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [7]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [8]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [9]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

client = genai.Client(
api_key=os.getenv("GOOGLE_API_KEY")
)


In [10]:
import json
user_prompt = json.dumps(doc)

In [11]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [13]:
from google.genai import types

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=user_prompt,
    config=types.GenerateContentConfig(
        system_instruction=data_gen_instructions,
        response_mime_type="application/json",
        response_schema=Questions,
    ),
)

In [15]:
questions = response.parsed
questions

Questions(questions=["Is it still possible to join the LLM Zoomcamp course if I'm starting late?", 'How can I receive a certificate for this course?', 'Do I need to complete a project to earn a course certificate?', 'What is the deadline for submitting the project to get the certificate?', 'Can I still participate in the course without submitting a project or aiming for a certificate?'])

In [16]:
from evaluation_utils import llm_structured

In [18]:
result, usage = llm_structured(
    client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

["Can I still sign up for the LLM Zoomcamp course if I'm late?", 'What are the main requirements for earning a certificate from this course?', 'Is submitting a project mandatory to receive the course certificate?', 'When is the final deadline for project submissions if I want to get a certificate?', 'If I enroll in the course now, can I still qualify for a certificate by submitting the project?']


In [19]:
usage

GenerateContentResponseUsageMetadata(
  candidates_token_count=105,
  prompt_token_count=174,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=174
    ),
  ],
  thoughts_token_count=1083,
  total_token_count=1362
)

In [20]:
from evaluation_utils import calc_price

In [21]:
calc_price(usage)

{'input_cost': 5.22e-05,
 'output_cost': 0.00026250000000000004,
 'total_cost': 0.00031470000000000006}

In [22]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': "Can I still sign up for the LLM Zoomcamp course if I'm late?",
  'document': '74eb249bbf'},
 {'question': 'What are the main requirements for earning a certificate from this course?',
  'document': '74eb249bbf'},
 {'question': 'Is submitting a project mandatory to receive the course certificate?',
  'document': '74eb249bbf'},
 {'question': 'When is the final deadline for project submissions if I want to get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'If I enroll in the course now, can I still qualify for a certificate by submitting the project?',
  'document': '74eb249bbf'}]

In [23]:
import pandas as pd

In [24]:
pd.DataFrame(records)

,question,document
0,Can I still sign up for the LLM Zoomcamp cours...,74eb249bbf
1,What are the main requirements for earning a c...,74eb249bbf
2,Is submitting a project mandatory to receive t...,74eb249bbf
3,When is the final deadline for project submiss...,74eb249bbf
4,"If I enroll in the course now, can I still qua...",74eb249bbf


In [25]:
from evaluation_utils import llm_structured_retry

In [28]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [29]:
generate_ground_truth(doc)

([{'question': 'Is it still possible to join the LLM Zoomcamp course at this point?',
   'document': '74eb249bbf'},
  {'question': 'What do I need to do to get a certificate for the course?',
   'document': '74eb249bbf'},
  {'question': 'Does completing a project have any bearing on receiving a certificate?',
   'document': '74eb249bbf'},
  {'question': "Is there a deadline for project submissions if I'm aiming for a certificate?",
   'document': '74eb249bbf'},
  {'question': 'Can I still receive a course completion certificate if I registered late?',
   'document': '74eb249bbf'}],
 GenerateContentResponseUsageMetadata(
   candidates_token_count=80,
   prompt_token_count=174,
   prompt_tokens_details=[
     ModalityTokenCount(
       modality=<MediaModality.TEXT: 'TEXT'>,
       token_count=174
     ),
   ],
   thoughts_token_count=645,
   total_token_count=899
 ))

In [30]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [31]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [32]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/115 [00:00<?, ?it/s]

In [33]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

575

In [34]:
ground_truth[10]

{'question': 'Will students get a Zoom link to join the live office hours?',
 'document': '489dd1c9d9'}

In [35]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.042916300000000004

In [36]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.042916300000000004

In [37]:
df_ground_truth = pd.DataFrame(ground_truth)

In [40]:
df_ground_truth.to_csv("../data/ground_truth.csv", index=False)

In [41]:
len(df_ground_truth)

575